# Retrieval Experiments

This notebook prepares retrieval experiments with the appropriate metadata, and runs the retrieval experiments.

## Section 0: Setup


### Imports


In [ ]:
import json
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
from tqdm.auto import tqdm

from openai import OpenAI
import anthropic
from google import genai
from google.genai import types as genai_types


from news_agent_utils.events import QUESTIONS_BY_EVENT
from news_agent_utils.io import read_table, save_table, stable_hash, utc_now_iso
from news_agent_utils.llm import (
    create_provider_client,
    load_provider_api_key,
    run_cached_tool_selection_call,
)

pd.set_option("display.max_colwidth", 160)


### Directories


In [ ]:
BASE_DIR = Path(".")
LLM_CACHE_DIR = BASE_DIR / "cache" / "retrieval" / "llm"
DATA_DIR = BASE_DIR / "data"

for path in [LLM_CACHE_DIR, DATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Experiment directory: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"LLM cache directory: {LLM_CACHE_DIR}")


### API Keys


In [ ]:
OPENAI_API_KEY = load_provider_api_key("openai")
ANTHROPIC_API_KEY = load_provider_api_key("anthropic")
GEMINI_API_KEY = load_provider_api_key("gemini")

OPENAI_CLIENT = create_provider_client("openai", OPENAI_API_KEY, openai_client_cls=OpenAI)
ANTHROPIC_CLIENT = create_provider_client("anthropic", ANTHROPIC_API_KEY, anthropic_module=anthropic)
GEMINI_CLIENT = create_provider_client("gemini", GEMINI_API_KEY, genai_module=genai)
PROVIDER_CLIENTS = {
    "openai": OPENAI_CLIENT,
    "anthropic": ANTHROPIC_CLIENT,
    "gemini": GEMINI_CLIENT,
}

print(f"Loaded OpenAI API key: {OPENAI_API_KEY is not None}.")
print(f"Loaded Anthropic API key: {ANTHROPIC_API_KEY is not None}.")
print(f"Loaded Gemini API key: {GEMINI_API_KEY is not None}.")
if not OPENAI_API_KEY:
    print("Set OPENAI_API_KEY before running OpenAI retrieval experiments.")
if not ANTHROPIC_API_KEY:
    print("Set ANTHROPIC_API_KEY before running Anthropic retrieval experiments.")
if not GEMINI_API_KEY:
    print("Set GEMINI_API_KEY before running Gemini retrieval experiments.")
if ANTHROPIC_API_KEY and anthropic is None:
    print("Anthropic key found, but the anthropic package is not importable.")
if GEMINI_API_KEY and genai is None:
    print("Gemini key found, but the google-genai package is not importable.")


### LLM Cache


In [ ]:
# Shared retrieval tool-selection LLM helpers are imported from news_agent_utils.llm.
LLM_CACHE_READ_ENABLED = True

## Section 1: Load Articles


In [ ]:
ARTICLE_TABLE_NAME = "df_articles"

df_articles_canonical = read_table(DATA_DIR, ARTICLE_TABLE_NAME)
if "article_id" not in df_articles_canonical.columns:
    raise ValueError("Article table must include an article_id column.")

df_articles = df_articles_canonical.copy().reset_index(drop=True)
if "hash_id" in df_articles.columns:
    df_articles = df_articles.drop(columns=["hash_id"])

df_articles = df_articles.rename(columns={"article_id": "hash_id"})
df_articles.insert(0, "article_id", [f"article_{idx + 1}" for idx in range(len(df_articles))])

print(f"Loaded {len(df_articles)} article rows across {df_articles['event'].nunique() if not df_articles.empty else 0} events from DATA_DIR / {ARTICLE_TABLE_NAME!r}.")
df_articles.columns


In [ ]:
EXPECTED_SLANTS = {"left", "center", "right"}

required_article_columns = {"article_id", "hash_id", "event", "slant", "source", "title", "body", "url"}
missing_article_columns = required_article_columns - set(df_articles.columns)
assert not missing_article_columns, f"Missing required article columns: {sorted(missing_article_columns)}"
assert set(df_articles["slant"].dropna().unique()).issubset(EXPECTED_SLANTS)
assert df_articles["article_id"].is_unique
assert df_articles["hash_id"].is_unique
assert df_articles["article_id"].str.match(r"^article_\d+$").all()
print(
    f"Loaded {len(df_articles)} article rows across "
    f"{df_articles['event'].nunique() if not df_articles.empty else 0} events. "
    "Canonical article_id is stored as hash_id; article_id is the LLM-facing sequential ID."
)


## Section 2: Define Questions


In [ ]:
# Shared retrieval queries are imported from news_agent_utils.events.
# Queries are sorted alphabetically within each event.
QUESTIONS_BY_EVENT


In [ ]:
article_events = set(df_articles["event"].unique())
query_events = set(QUESTIONS_BY_EVENT.keys())
missing_query_events = sorted(article_events - query_events)
assert not missing_query_events, f"Missing query lists for events: {missing_query_events}"

query_summary = pd.DataFrame(
    [
        {"event": event, "query_count": len(queries), "queries": queries}
        for event, queries in sorted(QUESTIONS_BY_EVENT.items())
        if event in article_events
    ]
)
query_summary


## Section 3: Retrieval Experiments


### Define the retrieval experiments

In [ ]:
RETRIEVAL_MODEL_SPECS = [
    {"model_provider": "openai", "model": "gpt-5.4"},
    {"model_provider": "anthropic", "model": "claude-opus-4-6"},
    {"model_provider": "gemini", "model": "gemini-3.1-pro-preview"},
]
RETRIEVAL_TEMPERATURE = None
RETRIEVAL_TOOL_SCHEMA_VERSION = "read_article_exact_20_v1"
METADATA_TYPES = ["source+headline", "headline", "source"]
RETRIEVAL_REPLICATES = 5
ARTICLES_TO_READ = 20
RUN_RETRIEVAL_EXPERIMENTS = True
RETRIEVAL_MAX_WORKERS = 3

SYSTEM_PROMPT_TEMPLATE = """You are a helpful news agent whose job is to select news articles that would help answer the user's query.

You should call `read_article` exactly once with a list of exactly {articles_to_read} different article IDs that are most relevant to the user's query.

Here is the list of articles you can read: {articles}"""

READ_ARTICLE_TOOL = {
    "type": "function",
    "name": "read_article",
    "description": f"Read exactly {ARTICLES_TO_READ} selected news articles by article_id.",
    "parameters": {
        "type": "object",
        "properties": {
            "article_ids": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": ARTICLES_TO_READ,
                "maxItems": ARTICLES_TO_READ,
                "description": f"A list of exactly {ARTICLES_TO_READ} different article_id values from the provided article list.",
            }
        },
        "required": ["article_ids"],
        "additionalProperties": False,
    },
}

valid_model_providers = {"openai", "anthropic", "gemini"}
for model_spec in RETRIEVAL_MODEL_SPECS:
    if set(model_spec) != {"model_provider", "model"}:
        raise ValueError(f"Each model spec must contain exactly model_provider and model: {model_spec}")
    if model_spec["model_provider"] not in valid_model_providers:
        raise ValueError(f"Unsupported model provider in RETRIEVAL_MODEL_SPECS: {model_spec}")
    if not str(model_spec["model"]).strip():
        raise ValueError(f"Model name cannot be empty: {model_spec}")

print(f"Using retrieval models: {RETRIEVAL_MODEL_SPECS}")
print(f"Metadata types: {METADATA_TYPES}")
print(f"Replicates per condition: {RETRIEVAL_REPLICATES}")
print(f"Articles to read per run: {ARTICLES_TO_READ}")
print(f"Retrieval API worker count: {RETRIEVAL_MAX_WORKERS}")


In [ ]:
def shuffle_event_articles(df_event_articles: pd.DataFrame, replicate: int) -> pd.DataFrame:
    """Shuffles the articles using the replicateas a seed."""
    records = df_event_articles.to_dict(orient="records")
    random.Random(replicate).shuffle(records) 
    return pd.DataFrame(records, columns=df_event_articles.columns)


def make_article_metadata(article: Dict[str, Any], metadata_type: str, include_hash_id: bool = False) -> Dict[str, Any]:
    """Constructs the metadata for the article that is presented to the LLM, based on the specified metadata type."""
    item = {"article_id": article["article_id"]}
    if include_hash_id:
        item["hash_id"] = article.get("hash_id", "")

    if metadata_type == "source+headline":
        item["source"] = article.get("source", "")
        item["headline"] = article.get("title", "")
        return item
    if metadata_type == "headline":
        item["headline"] = article.get("title", "")
        return item
    if metadata_type == "source":
        item["source"] = article.get("source", "")
        return item
    raise ValueError(f"Unsupported metadata_type: {metadata_type}")


def make_articles_metadata(
    df_event_articles: pd.DataFrame,
    metadata_type: str,
    include_hash_id: bool = False,
) -> List[Dict[str, Any]]:
    """Constructs the metadata for all articles that are presented to the LLM, based on the specified metadata type."""
    return [
        make_article_metadata(article, metadata_type, include_hash_id=include_hash_id)
        for article in df_event_articles.to_dict(orient="records")
    ]


def format_articles_for_prompt(prompt_metadata: List[Dict[str, Any]]) -> str:
    return json.dumps(prompt_metadata, ensure_ascii=False, indent=2)


def build_system_prompt(prompt_metadata: List[Dict[str, Any]], articles_to_read: int = ARTICLES_TO_READ) -> str:
    return SYSTEM_PROMPT_TEMPLATE.format(
        articles_to_read=articles_to_read,
        articles=format_articles_for_prompt(prompt_metadata),
    )


def build_retrieval_experiments_dataframe(
    df_articles: pd.DataFrame,
    questions_by_event: Dict[str, List[str]],
) -> pd.DataFrame:
    rows = []
    events = sorted(df_articles["event"].unique())
    queries_by_event = {
        event: [str(query).strip() for query in questions_by_event.get(event, []) if str(query).strip()]
        for event in events
    }
    expected_experiment_count = sum(
        len(queries_by_event[event]) * len(METADATA_TYPES) * RETRIEVAL_REPLICATES * len(RETRIEVAL_MODEL_SPECS)
        for event in events
    )

    with tqdm(total=expected_experiment_count, desc="Preparing retrieval experiments") as progress:
        for event in events:
            queries = queries_by_event[event]
            if not queries:
                continue

            df_event_articles = df_articles[df_articles["event"] == event].copy().reset_index(drop=True)
            candidate_article_count = len(df_event_articles)
            if candidate_article_count < ARTICLES_TO_READ:
                raise ValueError(
                    f"Event {event!r} has {candidate_article_count} article(s), "
                    f"but retrieval requires at least {ARTICLES_TO_READ}."
                )

            event_row_count = len(queries) * len(METADATA_TYPES) * RETRIEVAL_REPLICATES * len(RETRIEVAL_MODEL_SPECS)
            for query_index, query in enumerate(queries):
                for metadata_type in METADATA_TYPES:
                    for replicate in range(1, RETRIEVAL_REPLICATES + 1):
                        shuffle_seed = replicate
                        df_shuffled_articles = shuffle_event_articles(df_event_articles, shuffle_seed)
                        shuffled_article_ids = df_shuffled_articles["article_id"].tolist()
                        shuffled_hash_ids = df_shuffled_articles["hash_id"].tolist()
                        metadata = make_articles_metadata(df_shuffled_articles, metadata_type, include_hash_id=True)
                        prompt_metadata = make_articles_metadata(df_shuffled_articles, metadata_type, include_hash_id=False)
                        system_prompt = build_system_prompt(prompt_metadata, ARTICLES_TO_READ)
                        for model_spec in RETRIEVAL_MODEL_SPECS:
                            model_provider = model_spec["model_provider"]
                            model_name = model_spec["model"]
                            rows.append(
                                {
                                    "retrieval_experiment_id": stable_hash(
                                        {
                                            "event": event,
                                            "query_index": query_index,
                                            "query": query,
                                            "metadata_type": metadata_type,
                                            "replicate": replicate,
                                            "shuffled_article_ids": shuffled_article_ids,
                                            "shuffled_hash_ids": shuffled_hash_ids,
                                            "articles_to_read": ARTICLES_TO_READ,
                                            "model_provider": model_provider,
                                            "model": model_name,
                                        }
                                    ),
                                    "event": event,
                                    "query_index": query_index,
                                    "query": query,
                                    "metadata_type": metadata_type,
                                    "metadata": metadata,
                                    "prompt_metadata": prompt_metadata,
                                    "replicate": replicate,
                                    "shuffle_seed": shuffle_seed,
                                    "shuffled_article_ids": shuffled_article_ids,
                                    "shuffled_hash_ids": shuffled_hash_ids,
                                    "shuffled_articles_metadata": metadata,
                                    "system_prompt": system_prompt,
                                    "candidate_article_count": candidate_article_count,
                                    "articles_to_read": ARTICLES_TO_READ,
                                    "model_provider": model_provider,
                                    "model": model_name,
                                }
                            )
            progress.update(event_row_count)

    return pd.DataFrame(
        rows,
        columns=[
            "retrieval_experiment_id",
            "event",
            "query_index",
            "query",
            "metadata_type",
            "metadata",
            "prompt_metadata",
            "replicate",
            "shuffle_seed",
            "shuffled_article_ids",
            "shuffled_hash_ids",
            "shuffled_articles_metadata",
            "system_prompt",
            "candidate_article_count",
            "articles_to_read",
            "model_provider",
            "model",
        ],
    )


df_retrieval_experiments = build_retrieval_experiments_dataframe(df_articles, QUESTIONS_BY_EVENT)
save_table(df_retrieval_experiments, DATA_DIR, "df_retrieval_experiments")


In [ ]:
expected_experiment_columns = {
    "retrieval_experiment_id",
    "event",
    "query_index",
    "query",
    "metadata_type",
    "metadata",
    "prompt_metadata",
    "replicate",
    "shuffle_seed",
    "shuffled_article_ids",
    "shuffled_hash_ids",
    "shuffled_articles_metadata",
    "system_prompt",
    "candidate_article_count",
    "articles_to_read",
    "model_provider",
    "model",
}
assert expected_experiment_columns.issubset(df_retrieval_experiments.columns)

allowed_internal_metadata_keys = {
    "source+headline": {"article_id", "hash_id", "source", "headline"},
    "headline": {"article_id", "hash_id", "headline"},
    "source": {"article_id", "hash_id", "source"},
}
allowed_prompt_metadata_keys = {
    "source+headline": {"article_id", "source", "headline"},
    "headline": {"article_id", "headline"},
    "source": {"article_id", "source"},
}

if not df_retrieval_experiments.empty:
    assert df_retrieval_experiments["retrieval_experiment_id"].is_unique
    assert set(df_retrieval_experiments["metadata_type"].unique()) <= set(METADATA_TYPES)
    assert set(df_retrieval_experiments["replicate"].unique()) <= set(range(1, RETRIEVAL_REPLICATES + 1))
    assert set(df_retrieval_experiments["model_provider"].unique()) <= valid_model_providers
    assert set(df_retrieval_experiments[["model_provider", "model"]].itertuples(index=False, name=None)) <= {
        (model_spec["model_provider"], model_spec["model"])
        for model_spec in RETRIEVAL_MODEL_SPECS
    }
    assert (df_retrieval_experiments["candidate_article_count"] >= ARTICLES_TO_READ).all()
    assert (df_retrieval_experiments["articles_to_read"] == ARTICLES_TO_READ).all()

    expected_rows = 0
    for event in df_articles["event"].unique():
        query_count = len([query for query in QUESTIONS_BY_EVENT.get(event, []) if str(query).strip()])
        expected_rows += query_count * len(METADATA_TYPES) * RETRIEVAL_REPLICATES * len(RETRIEVAL_MODEL_SPECS)
    assert len(df_retrieval_experiments) == expected_rows

    for row in df_retrieval_experiments.to_dict(orient="records"):
        expected_internal_keys = allowed_internal_metadata_keys[row["metadata_type"]]
        expected_prompt_keys = allowed_prompt_metadata_keys[row["metadata_type"]]
        assert all(set(item.keys()) == expected_internal_keys for item in row["metadata"])
        assert all(set(item.keys()) == expected_prompt_keys for item in row["prompt_metadata"])
        assert row["metadata"] == row["shuffled_articles_metadata"]
        assert [item["article_id"] for item in row["metadata"]] == row["shuffled_article_ids"]
        assert [item["hash_id"] for item in row["metadata"]] == row["shuffled_hash_ids"]
        assert [item["article_id"] for item in row["prompt_metadata"]] == row["shuffled_article_ids"]
        assert all("hash_id" not in item for item in row["prompt_metadata"])
        assert "hash_id" not in row["system_prompt"]
        assert format_articles_for_prompt(row["prompt_metadata"]) in row["system_prompt"]

    grouped = df_retrieval_experiments.groupby(["event", "query", "metadata_type", "model_provider", "model"])
    for _, group in grouped:
        ordered = group.sort_values("replicate")
        repeat_seed = int(ordered.iloc[0]["replicate"])
        event = ordered.iloc[0]["event"]
        df_event_articles = df_articles[df_articles["event"] == event].copy().reset_index(drop=True)
        shuffled = shuffle_event_articles(df_event_articles, repeat_seed)
        assert shuffled["article_id"].tolist() == ordered.iloc[0]["shuffled_article_ids"]
        assert shuffled["hash_id"].tolist() == ordered.iloc[0]["shuffled_hash_ids"]
        if len(ordered) > 1:
            unique_orders = {tuple(ids) for ids in ordered["shuffled_article_ids"]}
            assert len(unique_orders) > 1

print(f"Prepared {len(df_retrieval_experiments)} retrieval experiment(s).")


### Tool Implementation


In [ ]:
def get_event_article_lookup(df_articles: pd.DataFrame, event: str) -> Dict[str, Dict[str, Any]]:
    df_event = df_articles[df_articles["event"] == event]
    return {row["article_id"]: row for row in df_event.to_dict(orient="records")}


def read_article(article_ids: List[str], article_lookup: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    if not isinstance(article_ids, list):
        return {"ok": False, "error": "article_ids must be a list.", "articles": []}

    normalized_ids = [str(article_id).strip() for article_id in article_ids]
    if len(normalized_ids) != ARTICLES_TO_READ:
        return {"ok": False, "error": f"read_article requires exactly {ARTICLES_TO_READ} article IDs.", "articles": []}
    if len(set(normalized_ids)) != len(normalized_ids):
        return {"ok": False, "error": "read_article requires different article IDs; duplicates were provided.", "articles": []}

    unknown_ids = [article_id for article_id in normalized_ids if article_id not in article_lookup]
    if unknown_ids:
        return {"ok": False, "error": f"Unknown article IDs: {unknown_ids}", "articles": []}

    articles = []
    for article_id in normalized_ids:
        article = article_lookup[article_id]
        articles.append(
            {
                "article_id": article["article_id"],
                "hash_id": article.get("hash_id", ""),
                "source": article.get("source", ""),
                "title": article.get("title", ""),
                "body": article.get("body", ""),
                "url": article.get("url", ""),
                "slant": article.get("slant", ""),
            }
        )
    return {"ok": True, "error": None, "articles": articles}


def parse_tool_arguments(arguments: Any) -> Dict[str, Any]:
    if isinstance(arguments, dict):
        return arguments
    if isinstance(arguments, str) and arguments.strip():
        return json.loads(arguments)
    return {}


def response_output_items(cache_row: Dict[str, Any]) -> List[Dict[str, Any]]:
    return cache_row.get("response", []) or []


In [ ]:
# Non-API validation for read_article.
if not df_articles.empty:
    eligible_events = [
        event for event, count in df_articles.groupby("event").size().items()
        if count >= ARTICLES_TO_READ
    ]
    if eligible_events:
        sample_event = eligible_events[0]
        sample_lookup = get_event_article_lookup(df_articles, sample_event)
        sample_ids = list(sample_lookup.keys())[:ARTICLES_TO_READ]

        assert not read_article(sample_ids[: ARTICLES_TO_READ - 1], sample_lookup)["ok"]
        assert not read_article(sample_ids + [sample_ids[0]], sample_lookup)["ok"]
        assert not read_article([sample_ids[0]] + sample_ids[:-1], sample_lookup)["ok"]
        assert not read_article(sample_ids[:-1] + ["missing-article-id"], sample_lookup)["ok"]
        valid_result = read_article(sample_ids, sample_lookup)
        assert valid_result["ok"]
        assert len(valid_result["articles"]) == ARTICLES_TO_READ
        assert {"article_id", "hash_id", "source", "title", "body", "url", "slant"}.issubset(valid_result["articles"][0].keys())

print("read_article validation checks passed.")


### Runner


In [ ]:
def build_selection_prompt(input_payload: List[Dict[str, Any]]) -> str:
    return json.dumps(input_payload, sort_keys=True, ensure_ascii=False)


def make_retrieval_selection_call_kwargs(experiment_row: Dict[str, Any]) -> Dict[str, Any]:
    input_payload = [
        {"role": "system", "content": experiment_row["system_prompt"]},
        {"role": "user", "content": experiment_row["query"]},
    ]
    return {
        "prompt": build_selection_prompt(input_payload),
        "input_payload": input_payload,
        "model_provider": experiment_row["model_provider"],
        "model_name": experiment_row["model"],
        "temperature": RETRIEVAL_TEMPERATURE,
        "tools": [READ_ARTICLE_TOOL],
        "output_mode": "read_article_selection",
    }


def build_retrieval_result_from_cache(
    experiment_row: Dict[str, Any],
    df_articles: pd.DataFrame,
    selection_cache_row: Dict[str, Any],
) -> Dict[str, Any]:
    event = experiment_row["event"]
    query = experiment_row["query"]
    metadata_type = experiment_row["metadata_type"]
    metadata = experiment_row["metadata"]
    prompt_metadata = experiment_row["prompt_metadata"]
    replicate = int(experiment_row["replicate"])
    shuffle_seed = int(experiment_row["shuffle_seed"])
    shuffled_article_ids = experiment_row["shuffled_article_ids"]
    shuffled_hash_ids = experiment_row["shuffled_hash_ids"]
    shuffled_articles_metadata = experiment_row["shuffled_articles_metadata"]
    system_prompt = experiment_row["system_prompt"]
    articles_to_read = int(experiment_row.get("articles_to_read", ARTICLES_TO_READ))
    model_provider = experiment_row["model_provider"]
    model_name = experiment_row["model"]

    article_lookup = get_event_article_lookup(df_articles, event)
    function_calls = [
        item for item in response_output_items(selection_cache_row)
        if item.get("type") == "function_call"
    ]
    tool_call_trace = []
    read_article_ids = []
    selected_articles = []
    validation_errors = []

    if len(function_calls) != 1:
        validation_errors.append(f"Expected exactly one read_article tool call; got {len(function_calls)}.")

    for tool_call in function_calls:
        tool_name = tool_call.get("name", "")
        call_id = tool_call.get("call_id")
        raw_arguments = tool_call.get("arguments", "{}")
        try:
            arguments = parse_tool_arguments(raw_arguments)
        except Exception as exc:
            arguments = {}
            validation_errors.append(f"Could not parse tool arguments: {exc}")

        if tool_name != "read_article":
            result = {"ok": False, "error": f"Unsupported tool: {tool_name}", "articles": []}
        else:
            article_ids = arguments.get("article_ids", [])
            read_article_ids = [str(article_id).strip() for article_id in article_ids] if isinstance(article_ids, list) else []
            result = read_article(article_ids, article_lookup)

        if not result.get("ok"):
            validation_errors.append(str(result.get("error") or "read_article failed."))
        selected_articles = result.get("articles", [])

        tool_call_trace.append(
            {
                "call_id": call_id,
                "name": tool_name,
                "arguments": arguments,
                "result_ok": result.get("ok"),
                "result_error": result.get("error"),
                "returned_article_ids": [article.get("article_id") for article in result.get("articles", [])],
            }
        )

    return {
        "retrieval_run_id": stable_hash(
            {
                "event": event,
                "query": query,
                "metadata_type": metadata_type,
                "replicate": replicate,
                "model_provider": model_provider,
                "model": model_name,
                "selection_llm_cache_key": selection_cache_row["cache_key"],
            }
        ),
        "event": event,
        "query": query,
        "metadata_type": metadata_type,
        "metadata": metadata,
        "prompt_metadata": prompt_metadata,
        "replicate": replicate,
        "shuffle_seed": shuffle_seed,
        "shuffled_article_ids": shuffled_article_ids,
        "shuffled_hash_ids": shuffled_hash_ids,
        "shuffled_articles_metadata": shuffled_articles_metadata,
        "system_prompt": system_prompt,
        "articles_to_read": articles_to_read,
        "model_provider": model_provider,
        "model": model_name,
        "selected_articles": selected_articles,
        "read_article_ids": read_article_ids,
        "read_article_count": len(set(read_article_ids)),
        "tool_call_trace": tool_call_trace,
        "valid_read_article_call": len(validation_errors) == 0,
        "validation_error": "; ".join(validation_errors),
        "selection_response_id": selection_cache_row.get("response_id"),
        "generated_at": utc_now_iso(),
        "selection_llm_cache_key": selection_cache_row["cache_key"],
    }


def run_retrieval_experiment(
    experiment_row: Dict[str, Any],
    df_articles: pd.DataFrame,
) -> Dict[str, Any]:
    selection_cache_row, cache_hit = run_cached_tool_selection_call(
        **make_retrieval_selection_call_kwargs(experiment_row),
        tool_schema_version=RETRIEVAL_TOOL_SCHEMA_VERSION,
        tool_name="read_article",
        llm_cache_dir=LLM_CACHE_DIR,
        provider_clients=PROVIDER_CLIENTS,
        genai_types=genai_types,
        cache_read_enabled=LLM_CACHE_READ_ENABLED,
    )
    result_row = build_retrieval_result_from_cache(experiment_row, df_articles, selection_cache_row)
    result_row["_llm_cache_hit"] = cache_hit
    return result_row


def run_retrieval_experiments(df_retrieval_experiments: pd.DataFrame, df_articles: pd.DataFrame) -> pd.DataFrame:
    experiment_records = df_retrieval_experiments.to_dict(orient="records")
    if not experiment_records:
        return pd.DataFrame(columns=globals().get("RESULT_COLUMNS", []))

    worker_count = max(1, min(int(RETRIEVAL_MAX_WORKERS), len(experiment_records)))
    print(f"Running {len(experiment_records)} retrieval experiment(s) with {worker_count} worker(s).")

    rows_by_index = {}
    if worker_count == 1:
        with tqdm(total=len(experiment_records), desc="Running retrieval experiments") as progress:
            for index, experiment_row in enumerate(experiment_records):
                if index % 10 == 0:
                    print(f"Running retrieval experiment {index} of {len(experiment_records)}")
                rows_by_index[index] = run_retrieval_experiment(experiment_row, df_articles)
                progress.update(1)
    else:
        with ThreadPoolExecutor(max_workers=worker_count) as executor:
            futures = {
                executor.submit(run_retrieval_experiment, experiment_row, df_articles): index
                for index, experiment_row in enumerate(experiment_records)
            }
            with tqdm(total=len(futures), desc="Running retrieval experiments") as progress:
                for completed_count, future in enumerate(as_completed(futures), start=1):
                    row_index = futures[future]
                    rows_by_index[row_index] = future.result()
                    if completed_count % 10 == 0 or completed_count == len(futures):
                        print(f"Completed retrieval experiment {completed_count} of {len(futures)}")
                    progress.update(1)

    rows = [rows_by_index[index] for index in sorted(rows_by_index)]
    cache_hit_count = sum(1 for row in rows if row.pop("_llm_cache_hit", False))
    provider_call_count = len(rows) - cache_hit_count
    print(
        "Retrieval cache summary: "
        f"{cache_hit_count} cache hit(s), "
        f"{provider_call_count} provider call(s)."
    )

    if not rows:
        return pd.DataFrame(columns=globals().get("RESULT_COLUMNS", []))
    return pd.DataFrame(rows)


In [ ]:
RESULT_COLUMNS = [
    "retrieval_run_id",
    "event",
    "query",
    "metadata_type",
    "metadata",
    "prompt_metadata",
    "replicate",
    "shuffle_seed",
    "shuffled_article_ids",
    "shuffled_hash_ids",
    "shuffled_articles_metadata",
    "system_prompt",
    "articles_to_read",
    "model_provider",
    "model",
    "selected_articles",
    "read_article_ids",
    "read_article_count",
    "tool_call_trace",
    "valid_read_article_call",
    "validation_error",
    "selection_response_id",
    "generated_at",
    "selection_llm_cache_key",
]

if RUN_RETRIEVAL_EXPERIMENTS:
    if df_retrieval_experiments.empty:
        raise ValueError("No retrieval experiments to run. Fill QUESTIONS_BY_EVENT first.")
    df_retrieval_results = run_retrieval_experiments(df_retrieval_experiments, df_articles)
else:
    print("RUN_RETRIEVAL_EXPERIMENTS is False. Set it to True after checking df_retrieval_experiments to call the API.")
    df_retrieval_results = pd.DataFrame(columns=RESULT_COLUMNS)

save_table(df_retrieval_results, DATA_DIR, "df_retrieval_results")


In [ ]:
expected_result_columns = set(RESULT_COLUMNS)
assert expected_result_columns.issubset(df_retrieval_results.columns)

if RUN_RETRIEVAL_EXPERIMENTS and not df_retrieval_experiments.empty:
    result_counts = (
        df_retrieval_results.groupby(["event", "query", "metadata_type", "replicate", "model_provider", "model"])
        .size()
        .reset_index(name="row_count")
    )
    assert (result_counts["row_count"] == 1).all()
    assert df_retrieval_results["retrieval_run_id"].is_unique
    assert (df_retrieval_results["articles_to_read"] == ARTICLES_TO_READ).all()

print("Retrieval validation checks passed.")
